# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irene501/flyrank/blob/main/work/notebooks/w04_signal_audit.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip install -q duckdb huggingface_hub pandas scikit-learn matplotlib

In [2]:
import os, getpass
import numpy as np
import pandas as pd
import duckdb

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':  f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':  f"read_parquet('{REL}/dim_content.parquet')",
    'fact_march':   f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

In [3]:
feature_frame = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions)                                            AS imp_h1,
        SUM(gsc_clicks)                                                 AS clicks_h1,
        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)               AS ctr_h1,
        AVG(gsc_avg_position) FILTER (WHERE gsc_avg_position > 0)       AS avg_position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0)  AS active_days_h1
    FROM {TABLES['fact_march']}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 10
""").df()

label_frame = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31') AS imp_h2
    FROM {TABLES['fact_march']}
    GROUP BY 1, 2
""").df()

data = feature_frame.merge(label_frame, on=["client_hash_id", "content_hash_id"], how="inner")
data["is_declining_proxy"] = (data["imp_h2"] < 0.8 * data["imp_h1"]).astype(int)
data = data.dropna(subset=["ctr_h1", "avg_position_h1"]).reset_index(drop=True)

def position_tier(p):
    if p <= 3: return "1-3"
    if p <= 10: return "4-10"
    if p <= 20: return "11-20"
    return "21+"

data["position_tier"] = data["avg_position_h1"].apply(position_tier)
print(f"{len(data):,} rows | is_declining_proxy rate: {data['is_declining_proxy'].mean():.3f}")
data.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

120,475 rows | is_declining_proxy rate: 0.296


,client_hash_id,content_hash_id,imp_h1,clicks_h1,ctr_h1,avg_position_h1,active_days_h1,imp_h2,is_declining_proxy,position_tier
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,57.0,0.0,0.000000,4.222711,15,20.0,1,4-10
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,199.0,2.0,0.010050,4.086084,13,403.0,0,4-10
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,467.0,1.0,0.002141,4.449176,13,343.0,1,4-10
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,56.0,0.0,0.000000,7.700694,14,26.0,1,4-10
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,771.0,1.0,0.001297,1.883472,14,1087.0,0,1-3


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [4]:
print(data[["imp_h1", "clicks_h1", "ctr_h1", "avg_position_h1", "active_days_h1"]].describe())


              imp_h1      clicks_h1         ctr_h1  avg_position_h1  \
count  120475.000000  120475.000000  120475.000000    120475.000000   
mean     1057.476564       3.186869       0.003010        16.225084   
std      2967.326193      14.800922       0.008678        16.906887   
min        10.000000       0.000000       0.000000         0.040763   
25%        55.000000       0.000000       0.000000         5.145136   
50%       216.000000       0.000000       0.000000         9.033372   
75%       856.000000       2.000000       0.002971        21.418724   
max    161575.000000    2395.000000       0.300000       127.620709   

       active_days_h1  
count   120475.000000  
mean        12.965769  
std          3.180422  
min          1.000000  
25%         12.000000  
50%         15.000000  
75%         15.000000  
max         15.000000  


Heavy tails: imp_h1 and clicks_h1 are the most heavily skewed — imp_h1's mean (1,057) is nearly 5x its median (216), with a max of 161,575, so a handful of very high-traffic pages are pulling the average way up. clicks_h1 is even more extreme: the median is 0 (most pages get zero clicks in this window) while the mean is 3.19, again dragged up by a max of 2,395. ctr_h1 shows the same pattern for the same reason — median 0, mean 0.003. avg_position_h1 is moderately skewed (mean 16.2 vs. median 9.0) but far less extreme, since position is naturally bounded and can't run away the way impression counts can. active_days_h1 is the exception — mean (13.0) is actually below the median (15), so it's slightly left-skewed rather than heavy-tailed, which makes sense since it's capped at 15 possible days in this window and piles up near that ceiling.

Practically: because imp_h1 and clicks_h1 are so skewed, any summary built from them should lean on the median, not the mean — same reasoning already applied in the flag-linked test.

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*



**Signal 1 — does `is_declining_proxy` rate fall as position gets worse, or improve?**
(if declines cluster in bad positions, position is a genuinely useful feature, not noise)

In [5]:
signal1 = data.groupby("position_tier")["is_declining_proxy"].agg(["mean", "count"]).reindex(["1-3", "4-10", "11-20", "21+"])
print(signal1)


                   mean  count
position_tier                 
1-3            0.249953  10582
4-10           0.320753  53786
11-20          0.275206  23804
21+            0.286785  32303


##Signal 1 (position vs decline) — verdict: MIXED.
Not a clean gradient: 1-3 → 0.250, 4-10 → 0.321, 11-20 → 0.275, 21+ → 0.287. Top positions (1-3) do decline less than everything else, but 4-10 is actually the worst tier, not the best-ranked-minus-one you'd expect. Position alone isn't a clean signal.

**Signal 2 — does `active_days_h1` (how often a page shows up at all) relate to decline rate?**
(a page active fewer days in h1 might already be fading before h2 confirms it)

In [6]:
data["active_days_tier"] = pd.cut(data["active_days_h1"], bins=[-1, 5, 10, 15], labels=["1-5", "6-10", "11-15"])
signal2 = data.groupby("active_days_tier", observed=True)["is_declining_proxy"].agg(["mean", "count"])
print(signal2)


                      mean  count
active_days_tier                 
1-5               0.203916   6282
6-10              0.321813  17013
11-15             0.297963  97180


##Signal 2 (active days vs decline) — verdict: OPPOSITE of the hypothesis.
 1-5 days → 0.204, 6-10 → 0.322, 11-15 → 0.298. The hypothesis was "shows up less often = already fading = more likely to decline." The data says the opposite at the low end — pages active only 1-5 days have the lowest decline rate, not the highest. (Plausible read: a page barely active in h1 has little impression volume left to lose, so it can't "decline" much further — a floor effect, not resilience.)

**Signal 3 — does `ctr_h1` (click-through rate in the first half of March) relate to decline rate?**
(a page already under-converting its impressions might be more likely to keep losing them)

In [7]:
data["ctr_tier"] = pd.qcut(data["ctr_h1"], q=4, duplicates="drop")
signal3 = data.groupby("ctr_tier", observed=True)["is_declining_proxy"].agg(["mean", "count"])
print(signal3)
# TODO once run: verdict -- CONFIRMED / OPPOSITE / MIXED / FALSE, one sentence why.

                       mean  count
ctr_tier                          
(-0.001, 0.00297]  0.325302  90356
(0.00297, 0.3]     0.209801  30119


##Signal 3 (CTR vs decline) — verdict: CONFIRMED , cleanly.
Bottom CTR quartile → 0.325 decline rate; top quartile → 0.210. Monotonic, clear gap. This is the strongest signal of the three.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [8]:
quick_win_check = data.groupby("position_tier")["imp_h1"].agg(["mean", "median", "count"]).reindex(["1-3", "4-10", "11-20", "21+"])
print(quick_win_check)


                      mean  median  count
position_tier                            
1-3            1870.525137   740.0  10582
4-10           1145.259826   295.0  53786
11-20           632.347379   188.0  23804
21+             958.246912    92.0  32303


##Flag-linked test (quick-win zone) — verdict: CONFIRMED.
 Median impressions: 1-3 → 740, 4-10 → 295, 11-20 → 188, 21+ → 92. The quick-win rule's assumption holds — pages ranked 11-20 do carry meaningfully more volume than 21+, so pushing them toward page one is a reasonable priority (note it's the median driving this read, not the mean — means are inflated by a few huge outliers, per the describe() output).

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

CTR is the one clean, trustworthy signal here — low-CTR pages decline at roughly 1.5x the rate of high-CTR ones, and it's monotonic across the whole range. Position on its own is weaker than expected — only the very top tier is meaningfully protected, and the middle tiers don't follow a clean pattern, so it shouldn't be trusted alone. Active-days behaved backwards from the hypothesis, which is a useful negative result: a page being quiet isn't the same as a page declining — it may just have less traffic left to lose. The quick-win rule's volume assumption checks out, so that part of the baseline is standing on solid ground.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.